In [0]:
!pip install -r /Workspace/Repos/operacion-maker/pacifico-metadata-ai/requirements.txt

In [0]:
import os
import time
import mlflow
from mlflow.models import infer_signature
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()
from agent import MetadataGovernanceAgent

In [0]:
mlflow.set_registry_uri("databricks-uc")
# Define tu catálogo y esquema
catalog_name = "workspace"  # <-- Cambia esto si usas otro catálogo
schema_name = "default"  # <-- Cambia esto si usas otro esquema
model_name = f"{catalog_name}.{schema_name}.metabuilder_agent"
endpoint_name = "metabuilder-endpoint"

print(f"Model Name: {model_name}")
print(f"Endpoint Name: {endpoint_name}")

In [0]:
# 1. Definir el esquema base para el agente
input_example = {
    "messages": [{"role": "user", "content": "main.default.my_table"}],
    "custom_inputs": {"thread_id": "12345"},
}

# 2. Inferir la firma explícitamente usando el ejemplo y un output de texto
signature = infer_signature(
    model_input=input_example,
    model_output=["Respuesta generada por el agente"]
)

with mlflow.start_run() as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="agent_model",
        python_model="agent.py", 
        code_path=[
            "agent.py", "graph", "state", "nodes", 
            "prompts", "tools", "config", "context", "routers"
        ],
        pip_requirements="requirements.txt", 
        input_example=input_example,
        signature=signature # <--- CRÍTICO: Esto habilita el Databricks Agent Framework
    )

print(f"Modelo registrado localmente en: {model_info.model_uri}")

# Registrar el modelo en Unity Catalog
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri, name=model_name
)

print(f"✅ Modelo {model_name} versión {registered_model.version} registrado en Unity Catalog.")

In [0]:
from mlflow.deployments import get_deploy_client

# Inicializamos el cliente de despliegue apuntando al entorno actual de Databricks
client = get_deploy_client("databricks")

endpoint_config = {
    "served_entities": [
        {
            "entity_name": model_name,
            "entity_version": registered_model.version, # La versión que acabas de registrar
            "workload_size": "Small",
            "scale_to_zero_enabled": True,
            
            # 💡 Inyección de Secretos usando diccionarios puros
            "env_vars": [
                {
                    "env_var_name": "DATABRICKS_HOST",
                    "secret_scope": "metadatos_scope", # Tu Secret Scope
                    "secret_key": "db_host"            # Tu Secret Key
                },
                {
                    "env_var_name": "DATABRICKS_TOKEN",
                    "secret_scope": "metadatos_scope",
                    "secret_key": "db_token"
                },
                {
                    "env_var_name": "DATABRICKS_SQL_WAREHOUSE_ID",
                    "secret_scope": "metadatos_scope",
                    "secret_key": "db_warehouse_id" 
                }
            ]
        }
    ]
}

print(f"Desplegando / Actualizando el Serving Endpoint: {endpoint_name}...")

try:
    # Intenta crear el endpoint si no existe
    client.create_endpoint(name=endpoint_name, config=endpoint_config)
    print(f"✅ Endpoint '{endpoint_name}' creado exitosamente y aprovisionándose.")
except Exception as e:
    # Si el endpoint ya existe, capturamos el error y lo actualizamos
    if "RESOURCE_ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
        print(f"El endpoint '{endpoint_name}' ya existe. Lanzando actualización a la nueva versión...")
        client.update_endpoint(endpoint=endpoint_name, config=endpoint_config)
        print("✅ Endpoint actualizado exitosamente.")
    else:
        # Si es otro tipo de error, lo mostramos
        raise e